# REMNET Stage 3 — D3 External Validation Only

**Frozen-checkpoint evaluation on the locked RealWaste D3 Section 2 cohort.** It loads the five seed-matched REMNET ensembles and evaluates the same 3,497 D3 images for each seed. D1 original-test evaluation is not performed.

## 1. Configuration and inputs

Uses the user-attached Stage 3 checkpoint dataset and the D3 Section 2 RealWaste manifest. Loads the 15 seed-matched REMNET constituent checkpoints from `Section3/Classification_checkpoints/Remnet/` and evaluates the locked external cohort only. No training, fine-tuning, or adaptation is performed.

In [11]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/REMNET_D3_Ensemble_External_Validation')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'REMNET'
CONSTITUENTS = ('MobileNetV2','ResNet50','EfficientNet-Lite0')
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical Stage 2 REMNET ensemble prediction files.
# Each file, when present, must have source_path and predicted_class_id columns,
# and preferably all six prob_CLASS columns. They must come from Stage 2.
# Set None to discover files automatically in the attached private Kaggle input.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen three-constituent REMNET ensemble; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/REMNET_D3_Ensemble_External_Validation
Seeds: (7, 49, 112, 345, 476)
Frozen three-constituent REMNET ensemble; no training or fine-tuning.


## 2. Dependencies and strict ensemble-checkpoint inventory

Load exactly 15 Stage 2 REMNET constituent checkpoints: MobileNetV2, ResNet50, and EfficientNet-Lite0 for seeds 7, 49, 112, 345, and 476. The standalone ResNet50 checkpoints are not used for the REMNET ensemble.

In [12]:
# Load dependencies without downloading pretrained weights.
for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU inference selected')
print('torch:', torch.__version__, 'timm:', timm.__version__)

def only_directory_with_name(root, name):
    matches = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one directory {name!r} under {root}; found {matches}')
    return matches[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
STANDALONE_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Resnet_50')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')
# D3 Section 2 manifest and class mapping are read directly from the user's Kaggle input.
# There is no embedded or reconstructed manifest.
def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

CHECKPOINTS = {}
for seed in SEEDS:
    CHECKPOINTS[seed] = {}
    for name in CONSTITUENTS:
        # Exact original Stage 2 filenames, with optional Kaggle duplicate suffix (1).
        pattern = re.compile(rf'{re.escape(name)}_seed{seed}(?:\(\d+\))?\.pt', re.IGNORECASE)
        found = sorted(p for p in REMNET_CKPT_DIR.rglob('*.pt') if pattern.fullmatch(p.name))
        if len(found) != 1:
            raise RuntimeError(f'Expected one REMNET checkpoint for {name} seed {seed}, found: {found}')
        CHECKPOINTS[seed][name] = found[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(remnet_pts) == 15, f'Expected 15 REMNET constituent checkpoints, found {len(remnet_pts)}'
assert len({p for group in CHECKPOINTS.values() for p in group.values()}) == 15
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not any(p.is_relative_to(STANDALONE_CKPT_DIR) for group in CHECKPOINTS.values() for p in group.values())

# Stage 2 REMNET-only historical references; do not use standalone ResNet50 reports.
HISTORICAL_PER_SEED = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_Performance_Per_Seed.csv'))), None)
HISTORICAL_SUMMARY = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_5_Seed_Performance_Summary.csv'))), None)
print('Total .pt checkpoints:', len(all_pts), '| REMNET constituent checkpoints:', len(remnet_pts))
for seed, group in CHECKPOINTS.items():
    for name, path in group.items(): print('REMNET', seed, name, ':', path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)
print('Historical REMNET per-seed:', HISTORICAL_PER_SEED or 'not present')
print('Historical REMNET summary:', HISTORICAL_SUMMARY or 'not present')


GPU: Tesla T4
torch: 2.11.0+cu130 timm: 1.0.30
Total .pt checkpoints: 30 | REMNET constituent checkpoints: 15
REMNET 7 MobileNetV2 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed7.pt
REMNET 7 ResNet50 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/ResNet50_seed7.pt
REMNET 7 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed7.pt
REMNET 49 MobileNetV2 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed49.pt
REMNET 49 ResNet50 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/ResNet50_seed49.pt
REMNET 49 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed49.pt
REMNET 112 MobileNetV2 : /kaggle/input/dataset

## 3. Validate the locked D3 manifest and six-class RealWaste mapping

Validates the user-provided D3 Section 2 manifest, class mapping, image count, unique hashes, class indices, eligibility status, and locked external-cohort fingerprint.

In [13]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Load the attached RealWaste D3 dataset

The notebook reads the RealWaste images from the mounted Kaggle input and uses the D3 Section 2 manifest to select and verify the 3,497 evaluation images. No D1 original-test evaluation is performed in this D3-only workflow.

In [14]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. Resolve D3 images and verify every locked image hash

Resolve the 3,497 D3 manifest entries against the attached RealWaste image folders and verify each image's SHA-256 before inference.

In [15]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Restore the frozen REMNET ensemble and preprocessing

Load the three seed-matched frozen constituent state dictionaries and apply the configured 484×484 RGB preprocessing. No training, fine-tuning, or adaptation is performed.

In [16]:
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((484,484)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

class LockedImageDataset(Dataset):
    def __init__(self, df, kind):
        self.df = df.reset_index(drop=True)
        self.path_col = 'external_relative_path'
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            image = TEST_TRANSFORM(im.convert('RGB'))
        return image, int(CLASS_TO_ID[row['class']]), i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))

def construct_constituent(name):
    n = len(CLASSES)
    if name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == 'ResNet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == 'EfficientNet-Lite0':
        m = timm.create_model('efficientnet_lite0', pretrained=False, num_classes=n)
    else:
        raise ValueError(name)
    return m

def restore_checkpoint(seed):
    ensemble, metadata, digests = {}, {}, {}
    for name in CONSTITUENTS:
        path = CHECKPOINTS[seed][name]
        obj = torch.load(path,map_location='cpu',weights_only=True)
        assert isinstance(obj,dict) and 'state_dict' in obj, f'Unsupported Stage 2 checkpoint: {path}'
        assert str(obj.get('model')) == name, f'Constituent mismatch in {path.name}: {obj.get("model")}'
        assert int(obj.get('seed')) == seed, f'Seed mismatch in {path.name}'
        m = construct_constituent(name)
        m.load_state_dict(obj['state_dict'],strict=True)
        m.to(DEVICE).eval()
        for param in m.parameters(): param.requires_grad_(False)
        ensemble[name],metadata[name],digests[name] = m,obj,file_sha256(path)
    return ensemble,metadata,digests

def infer(ensemble,df,kind):
    labels, probabilities, positions = [],[],[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            images=images.to(DEVICE,non_blocking=True)
            # Exact Stage 2 REMNET rule: arithmetic mean of three softmax vectors.
            stacked = torch.stack([torch.softmax(ensemble[name](images),dim=1)
                                   for name in CONSTITUENTS],dim=0)
            prob=stacked.mean(dim=0)
            probabilities.append(prob.cpu().numpy().astype(np.float64))
            labels.append(y.numpy()); positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probabilities=np.concatenate(probabilities)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Image order changed during inference.'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Label order mismatch.'
    assert probabilities.shape==(len(df),6) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1),1,atol=2e-6)
    return labels,probabilities

_smoke, _smoke_meta, _smoke_hash = restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    batch=torch.zeros(1,3,484,484,device=DEVICE)
    assert all(tuple(_smoke[name](batch).shape)==(1,6) for name in CONSTITUENTS)
print('REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS')
del _smoke,_smoke_meta,_smoke_hash,batch
if DEVICE.type=='cuda':torch.cuda.empty_cache()


REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS


## 7. Optional historical Stage 2 REMNET prediction references

Historical per-image prediction files are not used for the D3 evaluation. The D3-only workflow records the frozen checkpoint hashes and evaluates the locked external cohort directly.

## 8. Metrics, predictions, and confusion-matrix writers

Every metric is computed from the same fixed image ordering. Precision, recall and F1 use **macro averaging across all six classes** (`zero_division=0`). All CSV metric fields are stored as percentages.

In [17]:
METRICS=('accuracy','macro_precision','macro_recall','macro_f1')
PRED_DIR=OUTPUT/'Predictions'; PRED_DIR.mkdir(exist_ok=True)
CM_DIR=OUTPUT/'Confusion_Matrices'; CM_DIR.mkdir(exist_ok=True)

def measure(seed,kind,df,y_true,prob,checkpoint_sha,verification_status):
    y_pred=prob.argmax(axis=1).astype(int)
    precision,recall,f1,_=precision_recall_fscore_support(
        y_true,y_pred,labels=list(range(6)),average='macro',zero_division=0)
    scores={'model':MODEL_NAME,'seed':seed,'dataset':kind,'n_images':len(df),
            'accuracy':100*accuracy_score(y_true,y_pred),
            'macro_precision':100*precision,
            'macro_recall':100*recall,'macro_f1':100*f1,
            'checkpoint_sha256':checkpoint_sha,
            'external_manifest_fingerprint':EXTERNAL_FINGERPRINT}
    return scores,y_pred

def save_predictions(seed,kind,df,y_true,y_pred,prob,checkpoint_sha):
    relative='source_path' if kind=='Original463' else 'external_relative_path'
    out=pd.DataFrame({
        'model':MODEL_NAME,'seed':seed,'dataset':kind,
        'image_relative_path':df[relative].astype(str).tolist(),
        'image_sha256':df['sha256'].tolist(),
        'true_class_id':y_true,
        'true_class':[CLASSES[i] for i in y_true],
        'predicted_class_id':y_pred,
        'predicted_class':[CLASSES[i] for i in y_pred],
        'prediction_confidence':prob.max(axis=1),
        'checkpoint_sha256':checkpoint_sha})
    for i,c in enumerate(CLASSES): out[f'prob_{c}']=prob[:,i]
    assert len(out)==len(df) and out['image_relative_path'].is_unique
    destination=PRED_DIR/f'REMNET_seed{seed}_{kind}_probabilities.csv'
    out.to_csv(destination,index=False,float_format='%.10g')
    return destination

def save_per_class(seed,kind,y_true,y_pred):
    values=classification_report(y_true,y_pred,labels=list(range(6)),target_names=list(CLASSES),
                                 output_dict=True,zero_division=0)
    records=[]
    for cls in CLASSES:
        r=values[cls]
        records.append({'model':MODEL_NAME,'seed':seed,'dataset':kind,'class':cls,
                        'output_index':CLASS_TO_ID[cls],'support':int(r['support']),
                        'precision_pct':100*r['precision'],
                        'recall_pct':100*r['recall'],'f1_pct':100*r['f1-score']})
    path=OUTPUT/f'REMNET_seed{seed}_{kind}_per_class.csv'
    pd.DataFrame(records).to_csv(path,index=False,float_format='%.10f')
    return path

def save_confusion(seed,kind,y_true,y_pred):
    cm=confusion_matrix(y_true,y_pred,labels=list(range(6)))
    path=CM_DIR/f'REMNET_seed{seed}_{kind}_confusion.csv'
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(path)
    fig,ax=plt.subplots(figsize=(7,6))
    ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(6),labels=CLASSES,rotation=45,ha='right')
    ax.set_yticks(range(6),labels=CLASSES)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(f'REMNET | seed {seed} | {kind} | n={len(y_true)}')
    threshold=cm.max()/2
    for i in range(6):
        for j in range(6):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>threshold else 'black',fontsize=9)
    fig.tight_layout()
    fig.savefig(CM_DIR/f'REMNET_seed{seed}_{kind}_confusion.png',dpi=160)
    plt.close(fig)
    return path

## 8. Run five frozen seed-matched REMNET ensembles on the 3,497-image D3 cohort

For each seed, load the three original frozen REMNET constituent checkpoints and evaluate the same locked D3 images. No D1 test-set inference or historical original-test comparison is performed.

In [18]:
per_seed=[]
checkpoint_registry=[]
for seed in SEEDS:
    print(f'\n=== REMNET ENSEMBLE SEED {seed} ===',flush=True)
    ensemble, metadata, hashes = restore_checkpoint(seed)
    for name in CONSTITUENTS:
        checkpoint_registry.append({
            'model':MODEL_NAME,'seed':seed,'constituent':name,
            'checkpoint_path':str(CHECKPOINTS[seed][name]),
            'checkpoint_sha256':hashes[name],
            'checkpoint_best_epoch':metadata[name].get('best_epoch'),
            'framework':('timm' if name=='EfficientNet-Lite0' else 'torchvision'),
            'num_output_classes':6,'input_size':'484x484',
            'normalization_mean':'0.5,0.5,0.5','normalization_std':'0.5,0.5,0.5',
            'external_manifest_fingerprint':EXTERNAL_FINGERPRINT})
    ensemble_sha=json.dumps(hashes,sort_keys=True)
    y_true_ext,p_ext=infer(ensemble,external_eval_df,'external')
    ext_score,pred_ext=measure(seed,'External3497',external_eval_df,y_true_ext,p_ext,ensemble_sha,'NOT_APPLICABLE_D3_ONLY')
    per_seed.append(ext_score)
    save_predictions(seed,'External3497',external_eval_df,y_true_ext,pred_ext,p_ext,ensemble_sha)
    save_per_class(seed,'External3497',y_true_ext,pred_ext)
    save_confusion(seed,'External3497',y_true_ext,pred_ext)
    print('External n:',len(y_true_ext),'| Accuracy:',round(ext_score['accuracy'],4),
          '| Macro-F1:',round(ext_score['macro_f1'],4))
    del ensemble
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

results_df=pd.DataFrame(per_seed)
registry_df=pd.DataFrame(checkpoint_registry)
results_df.to_csv(OUTPUT/'REMNET_All_Seeds_D3_External.csv',index=False,float_format='%.10f')
registry_df.to_csv(OUTPUT/'REMNET_Checkpoint_Registry.csv',index=False)
assert len(registry_df)==15 and len(results_df)==5
assert results_df.groupby('dataset').size().to_dict()=={'External3497':5}
print('\nPER-SEED REMNET D3 RESULTS:')
print(results_df[['seed','dataset','n_images',*METRICS]].to_string(index=False,float_format=lambda v:f'{v:.4f}'))


=== REMNET ENSEMBLE SEED 7 ===
External n: 3497 | Accuracy: 46.0967 | Macro-F1: 41.1166

=== REMNET ENSEMBLE SEED 49 ===
External n: 3497 | Accuracy: 39.5768 | Macro-F1: 35.3473

=== REMNET ENSEMBLE SEED 112 ===
External n: 3497 | Accuracy: 47.2119 | Macro-F1: 43.6128

=== REMNET ENSEMBLE SEED 345 ===
External n: 3497 | Accuracy: 42.9511 | Macro-F1: 37.9123

=== REMNET ENSEMBLE SEED 476 ===
External n: 3497 | Accuracy: 48.0126 | Macro-F1: 45.2521

PER-SEED REMNET D3 RESULTS:
 seed      dataset  n_images  accuracy  macro_precision  macro_recall  macro_f1
    7 External3497      3497   46.0967          53.0739       43.6272   41.1166
   49 External3497      3497   39.5768          47.5330       37.7604   35.3473
  112 External3497      3497   47.2119          50.5105       45.1488   43.6128
  345 External3497      3497   42.9511          56.3344       40.9068   37.9123
  476 External3497      3497   48.0126          56.6868       46.3597   45.2521


## 9. Five-seed mean ± sample standard deviation on D3

Summarizes the five independent seed-matched frozen REMNET ensemble evaluations on the same locked 3,497-image external cohort.

In [19]:
summary=[]
for metric in METRICS:
    values=results_df.set_index('seed').loc[list(SEEDS),metric]
    summary.append({
        'model':MODEL_NAME,'metric':metric,'n_seeds':5,
        'external_images':3497,
        'external_mean_pct':values.mean(),
        'external_sample_sd_pct':values.std(ddof=1)})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(OUTPUT/'REMNET_Five_Seed_Mean_SD_D3_External.csv',
                  index=False,float_format='%.10f')
print('REMNET — D3 EXTERNAL (3497 images)')
for row in summary_df.itertuples(index=False):
    print(f'{row.metric:17s}: {row.external_mean_pct:.2f} +/- {row.external_sample_sd_pct:.2f}%')

REMNET — D3 EXTERNAL (3497 images)
accuracy         : 44.77 +/- 3.48%
macro_precision  : 52.83 +/- 3.89%
macro_recall     : 42.76 +/- 3.46%
macro_f1         : 40.65 +/- 4.06%


## 10. Export D3 predictions, metrics, and checkpoint identities

Exports five-seed D3 predictions and probabilities, per-class metrics, confusion matrices, mean ± sample SD, and SHA-256 identities of all 15 REMNET constituent checkpoints.

In [20]:
report={
    'experiment':'REMNET Stage 3 D3-only external classification evaluation',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_notebook':'Remnet_Accu_s2(3).ipynb',
    'checkpoint_source':'Stage 2 REMNET three seed-matched best-validation-loss constituent checkpoints',
    'checkpoint_dataset_input':str(INPUT_ROOT),'checkpoint_seeds':list(SEEDS),
    'stage3_external_dataset':EXTERNAL_HANDLE,
    'd3_manifest_csv_path':str(EXTERNAL_MANIFEST),
    'stage3_external_dataset_version':'Kaggle RealWaste mounted copy; version unverified',
    'external_download_method':external_download_method,
    'external_resolved_path':str(external_download),
    'external_evaluation_count':3497,
    'd1_original_test_evaluation_performed':False,
    'external_locked_manifest_fingerprint_sha256':EXTERNAL_FINGERPRINT,
    'class_to_idx':CLASS_TO_ID,
    'external_transform':'RGB,Resize(484,484),ToTensor,Normalize(mean=.5x3,std=.5x3)',
    'model_constructor':'torchvision MobileNetV2 and ResNet50; timm EfficientNet-Lite0 (each six outputs)',
    'model_loading':'three strict-loaded seed-matched constituent state_dicts; mean of three per-class softmax vectors; eval; inference_mode',
    'training_or_adaptation_performed':False,
    'checkpoint_selection_on_external_dataset':False,
    'external_subset_shared_across_models':'D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv',
    'ensemble_constituents':list(CONSTITUENTS),
    'ensemble_rule':'equal-weight arithmetic mean of three per-class Softmax probability vectors',
    'external_3497_test_sha256_verified':True,
    'same_checkpoints_for_future_energy_inference':'REMNET_Checkpoint_Registry.csv: match all three constituent SHA-256s per seed',
    'torch':torch.__version__,
    'torchvision':__import__('torchvision').__version__,
    'device':str(DEVICE),
    'limitations':[
        'No D1 original-test results or D1-versus-D3 generalization gap are reported.',
        'Automated conservative screening does not demonstrate independent dataset acquisition.'
    ]}
(OUTPUT/'REMNET_External_Validation_Configuration.json').write_text(
    json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')

archive=OUTPUT/'REMNET_D3_REMNET_External_Only_Outputs.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as outzip:
    for f in sorted(OUTPUT.rglob('*')):
        if f.is_file() and f!=archive and 'original_source_extracted' not in f.parts:
            outzip.write(f,f.relative_to(OUTPUT))
print('OUTPUTS SAVED:',OUTPUT)
print('ZIP:',archive)
print('STAGE 3 D3-ONLY REMNET CLASSIFICATION EVALUATION COMPLETE: True')

OUTPUTS SAVED: /kaggle/working/REMNET_D3_Ensemble_External_Validation
ZIP: /kaggle/working/REMNET_D3_Ensemble_External_Validation/REMNET_D3_REMNET_External_Only_Outputs.zip
STAGE 3 D3-ONLY REMNET CLASSIFICATION EVALUATION COMPLETE: True
